In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from collections import defaultdict

In [ ]:
df = pd.read_excel('ligaMX_dataset_v1.xlsx', index_col = 0)
df.to_csv('LigaMX_dataset_v1.csv', index = True)

In [ ]:
df = pd.read_csv('LigaMX_dataset_v1.csv', index_col = 0)

In [ ]:
df.head()

,Date,Time,Round,Day,Venue,Opponent,Referee,Equipo,Torneo,Temporada,Result
NaN,2025-07-12,19:00,Jornada 1,Sat,Home,Necaxa,Fernando Hernández,Toluca,Apertura,2025,W
NaN,2025-07-16,21:05,Jornada 2,Wed,Away,Santos Laguna,Salvador Pérez Villalobos,Toluca,Apertura,2025,W
NaN,2025-07-26,19:00,Jornada 3,Sat,Home,UANL,Marco Antonio Ortíz,Toluca,Apertura,2025,L
NaN,2025-08-11,21:00,Jornada 4,Mon,Away,FC Juárez,Katia García,Toluca,Apertura,2025,W
NaN,2025-08-16,21:10,Jornada 5,Sat,Home,UNAM,Víctor Cáceres,Toluca,Apertura,2025,D


In [ ]:
df.shape

(7332, 11)

In [ ]:
df.dtypes

,0
Date,object
Time,object
Round,object
Day,object
Venue,object
Opponent,object
Referee,object
Equipo,object
Torneo,object
Temporada,int64


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

In [ ]:
opponent_name_fix = {
    'América': 'America',
    'Atlético San Luis': 'Atletico San Luis',
    'FC Juárez': 'FC Juarez',
    'León': 'Leon',
    'Mazatlán': 'Mazatlan',
    'Querétaro': 'Queretaro',
    'UANL': 'Tigres UANL',
    'UNAM': 'Pumas UNAM',
}
df['Opponent'] = df['Opponent'].replace(opponent_name_fix)

In [ ]:
df = df.sort_values(['Date', 'Equipo']).reset_index(drop=True)

In [ ]:
df["VenueID"] = df["Venue"].astype("category").cat.codes

In [ ]:
df["OpponentID"] = df["Opponent"].astype('category').cat.codes
df["TeamID"] = df["Equipo"].astype('category').cat.codes
df["RefereeID"] = df["Referee"].astype('category').cat.codes
df["RoundID"] = df["Round"].astype('category').cat.codes
df["TemporadaID"] = df["Temporada"].astype('category').cat.codes
df["TorneoID"] = df["Torneo"].astype('category').cat.codes

In [ ]:
df["TimeID"] = df["Time"].str.replace(":.+","", regex=True).astype("int")

In [ ]:
df["DayID"] = df["Date"].dt.dayofweek

In [ ]:
df["ResultID"] = df["Result"].map({"W": 1, "L": 0, "D": 2}).astype("int")

In [ ]:
K = 32          # K-factor (how quickly ratings adjust)
START_ELO = 1500  # Starting Elo for all teams
HOME_ADVANTAGE = 50  # Home team Elo bonus

elo_ratings = defaultdict(lambda: START_ELO)
elo_history = []

In [ ]:
df['match_key'] = df.apply(
    lambda r: tuple(sorted([r['Equipo'], r['Opponent']])) + (r['Date'],), axis=1
)

processed_matches = set()
row_elos = {}  # index -> (team_elo, opponent_elo)

for idx, row in df.iterrows():
    team = row['Equipo']
    opponent = row['Opponent']
    match_key = row['match_key']

    # Record pre-match Elo for this row
    row_elos[idx] = (elo_ratings[team], elo_ratings[opponent])

    # Only update Elo once per match
    if match_key not in processed_matches:
        processed_matches.add(match_key)

        team_elo = elo_ratings[team]
        opp_elo = elo_ratings[opponent]

        # Apply home advantage
        if row['Venue'] == 'Home':
            adjusted_team_elo = team_elo + HOME_ADVANTAGE
            adjusted_opp_elo = opp_elo
        else:
            adjusted_team_elo = team_elo
            adjusted_opp_elo = opp_elo + HOME_ADVANTAGE

        # Expected scores
        exp_team = 1 / (1 + 10 ** ((adjusted_opp_elo - adjusted_team_elo) / 400))
        exp_opp = 1 - exp_team

        # Actual scores
        if row['Result'] == 'W':
            actual_team, actual_opp = 1.0, 0.0
        elif row['Result'] == 'L':
            actual_team, actual_opp = 0.0, 1.0
        else:  # Draw
            actual_team, actual_opp = 0.5, 0.5

        # Update Elo
        elo_ratings[team] += K * (actual_team - exp_team)
        elo_ratings[opponent] += K * (actual_opp - exp_opp)

# Assign Elo columns
df['TeamElo'] = df.index.map(lambda i: round(row_elos[i][0], 2))
df['OponentElo'] = df.index.map(lambda i: round(row_elos[i][1], 2))
df['EloDiff'] = df['TeamElo'] - df['OponentElo']


In [ ]:
points_map = {'W': 3, 'D': 1, 'L': 0}
df['Points'] = df['Result'].map(points_map)

WINDOW = 5  # Last N games

def compute_rolling_features(group):
    """Compute rolling form features for a single team."""
    group = group.sort_values('Date')

    # Overall form - last 5 games points
    group['TeamForm5'] = group['Points'].shift(1).rolling(WINDOW, min_periods=1).sum()

    # Win rate last 5
    group['TeamWinRate5'] = (group['Result'].shift(1) == 'W').rolling(WINDOW, min_periods=1).mean()

    # Season cumulative points
    season_groups = []
    for (temporada, torneo), season in group.groupby(['Temporada', 'Torneo'], sort=False):
        season = season.sort_values('Date')
        season['TeamSeasonPts'] = season['Points'].shift(1).cumsum().fillna(0)
        season['MatchDay'] = range(1, len(season) + 1)
        season_groups.append(season)
    group = pd.concat(season_groups).sort_values('Date')

    return group

frames = []
for team, group in df.groupby('Equipo'):
    frames.append(compute_rolling_features(group))
df = pd.concat(frames).sort_values(['Date', 'Equipo']).reset_index(drop=True)

In [ ]:
def compute_venue_form(group):
    """Compute home/away specific form."""
    group = group.sort_values('Date')

    home_mask = group['Venue'] == 'Home'
    away_mask = group['Venue'] == 'Away'

    # Home form - rolling sum of points in last 5 HOME games
    home_pts = group['Points'].where(home_mask, np.nan)
    group['TeamHomeForm5'] = home_pts.shift(1).rolling(WINDOW, min_periods=1).sum()

    # Away form - rolling sum of points in last 5 AWAY games
    away_pts = group['Points'].where(away_mask, np.nan)
    group['TeamAwayForm5'] = away_pts.shift(1).rolling(WINDOW, min_periods=1).sum()

    return group

frames = []
for team, group in df.groupby('Equipo'):
    frames.append(compute_venue_form(group))
df = pd.concat(frames).sort_values(['Date', 'Equipo']).reset_index(drop=True)

In [ ]:
team_form_lookup = df.set_index(['Equipo', 'Date'])[['TeamForm5', 'TeamWinRate5', 'TeamSeasonPts', 'TeamHomeForm5', 'TeamAwayForm5']]

def get_opponent_features(row):
    """Look up opponent's form features."""
    try:
        opp_data = team_form_lookup.loc[(row['Opponent'], row['Date'])]
        if isinstance(opp_data, pd.DataFrame):
            opp_data = opp_data.iloc[0]
        return pd.Series({
            'OpponentForm5': opp_data['TeamForm5'],
            'OpponentWinRate5': opp_data['TeamWinRate5'],
            'OpponentSeasonPts': opp_data['TeamSeasonPts'],
            'OpponentHomeForm5': opp_data['TeamHomeForm5'],
            'OpponentAwayForm5': opp_data['TeamAwayForm5'],
        })
    except KeyError:
        return pd.Series({
            'OpponentForm5': np.nan,
            'OpponentWinRate5': np.nan,
            'OpponentSeasonPts': np.nan,
            'OpponentHomeForm5': np.nan,
            'OpponentAwayForm5': np.nan,
        })

opp_features = df.apply(get_opponent_features, axis=1)
df = pd.concat([df, opp_features], axis=1)


In [ ]:
def compute_h2h(df):
    """Compute historical head-to-head record."""
    df = df.sort_values(['Date', 'Equipo']).reset_index(drop=True)

    h2h_wins = defaultdict(int)      # (team, opponent) -> wins count
    h2h_matches = defaultdict(int)   # (team, opponent) -> match count
    h2h_last5 = defaultdict(list)    # (team, opponent) -> list of last results

    h2h_winrate_col = []
    h2h_last5_col = []

    processed = set()

    for idx, row in df.iterrows():
        team = row['Equipo']
        opponent = row['Opponent']
        pair = (team, opponent)

        # Record pre-match H2H stats
        total = h2h_matches[pair]
        if total > 0:
            h2h_winrate_col.append(h2h_wins[pair] / total)
        else:
            h2h_winrate_col.append(np.nan)

        last5 = h2h_last5[pair][-5:]
        if len(last5) > 0:
            h2h_last5_col.append(sum(last5))
        else:
            h2h_last5_col.append(np.nan)

        # Update H2H after recording
        match_id = (min(team, opponent), max(team, opponent), row['Date'])
        if match_id not in processed:
            processed.add(match_id)

            h2h_matches[pair] += 1
            h2h_matches[(opponent, team)] += 1

            pts = points_map[row['Result']]
            opp_pts = points_map[{'W': 'L', 'L': 'W', 'D': 'D'}[row['Result']]]

            if row['Result'] == 'W':
                h2h_wins[pair] += 1
            elif row['Result'] == 'L':
                h2h_wins[(opponent, team)] += 1

            h2h_last5[pair].append(pts)
            h2h_last5[(opponent, team)].append(opp_pts)

    df['H2HWinRate'] = h2h_winrate_col
    df['H2HLast5'] = h2h_last5_col

    return df

df = compute_h2h(df)


In [ ]:
df['FormDiff'] = df['TeamForm5'] - df['OpponentForm5']
df['SeasonPtsDiff'] = df['TeamSeasonPts'] - df['OpponentSeasonPts']

In [ ]:
df.to_csv('LigaMX_dataset_v2.csv', index = True)